In [0]:
  # Gold aggregation — daily revenue by store
  # Author: oakville3456
  # Updated: 2026-06-04
from pyspark.sql import functions as F
from datetime import datetime, timezone

# ── config ──────────────────────────────────────────
RAW        = "abfss://raw-landing@saretailsalesdev.dfs.core.windows.net/sales/"
BRONZE     = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/sales"
CHECKPOINT = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_checkpoint/bronze_sales"
SCHEMA_LOC = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_schema/bronze_sales"
RUN_LOG    = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_run_log/bronze_sales"

run_start = datetime.now(timezone.utc)   # ← fixed

# ── snapshot BEFORE run ──────────────────────────────
try:
    rows_before = spark.read.format("delta").load(BRONZE).count()
except:
    rows_before = 0

# ── Auto Loader read ─────────────────────────────────
raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(RAW)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

# ── write to Bronze ──────────────────────────────────
query = (
    raw.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("adb_retail_dev.bronze.sales") 
)

query.awaitTermination()

# ── snapshot AFTER run ───────────────────────────────
bronze = spark.read.format("delta").load(BRONZE)
rows_after = bronze.count()
rows_added = rows_after - rows_before

# ── files processed in this run ──────────────────────
new_files = (
    bronze
    .filter(F.col("_ingested_at") >= F.lit(run_start))
    .select("_source_file")
    .distinct()
    .collect()
)
new_file_list = [r._source_file for r in new_files]

# ── write run log ────────────────────────────────────
log = spark.createDataFrame([{
    "run_timestamp":  run_start.isoformat(),
    "rows_before":    rows_before,
    "rows_after":     rows_after,
    "rows_added":     rows_added,
    "files_added":    len(new_file_list),
    "status":         "new_data" if rows_added > 0 else "no_new_data",
    "files":          str(new_file_list)
}])

log.write \
    .format("delta") \
    .mode("append") \
    .save(RUN_LOG)

# ── print summary ────────────────────────────────────
print(f"✅ Bronze rows before:  {rows_before}")
print(f"✅ Bronze rows after:   {rows_after}")
print(f"✅ Rows added:          {rows_added}")
print(f"✅ Files processed:     {len(new_file_list)}")
print(f"✅ Status: {'NEW DATA INGESTED' if rows_added > 0 else 'NO NEW DATA — skipped'}")